# Práctica 3 — Ejercicio 2
## Red convolucional sobre MNIST con **PyTorch**

El mismo problema del Ejercicio 1 (clasificar los dígitos de MNIST con una CNN),
pero ahora escrito en PyTorch, para poder comparar las dos sintaxis.

**Guion del enunciado** (cada punto está marcado en el código con `[n]`):

| | |
|---|---|
| `[1]` | crear la clase `NN` que hereda de `torch.nn.Module` |
| `[2]` | en `__init__` definir las capas combinando `Conv2d` y `Linear` |
| `[3]` | en `forward` combinar `relu` con `max_pool2d`, aplanar con `x.view` y salir con `log_softmax` |
| `[4]` | comparar esta sintaxis con la de Keras, especialmente el cálculo de dimensiones |
| `[5]` | definir batch size, número de clases, learning rate y número de épocas |
| `[6]` | cargar los datos con `torchvision.datasets.MNIST` y crear el `DataLoader` |
| `[7]` | instanciar el modelo, definir la función de pérdida y el optimizador |
| `[8]` | loop sobre épocas y loop sobre batches anidados |
| `[9]` | en cada paso: forward, pérdida, gradientes a 0, backward, `optimizer.step`, acumular |
| `[10]` | calcular las predicciones sobre los datos de test |
| `[11]` | *(opcional)* matriz de confusión con `sklearn.metrics` |

---

> ⚙️ **Antes de correr:** activá la GPU en
> **Entorno de ejecución → Cambiar tipo de entorno de ejecución → Acelerador por hardware: GPU (T4)**.
> Con GPU cada época tarda ~8 s; en CPU, alrededor de un minuto.
>
> Después, **Entorno de ejecución → Ejecutar todo**.

## 0. Imports y hardware disponible

En Colab ya vienen instalados `torch`, `torchvision`, `sklearn` y `matplotlib`,
así que no hace falta instalar nada.

A diferencia de Keras, en PyTorch **el manejo del dispositivo es explícito**: hay
que mover a mano el modelo y cada batch a la GPU con `.to(device)`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from torchvision import datasets, transforms

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

from tqdm.auto import tqdm

# torch corre en CPU o GPU segun lo que haya disponible.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"torch   : {torch.__version__}")
print(f"device  : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
else:
    print("!! Sin GPU: Entorno de ejecucion -> Cambiar tipo de entorno de ejecucion -> GPU")

## 1. Hiperparámetros `[5]`

In [ ]:
SEED = 0

BATCH_SIZE    = 128     # [5] mismo batch que en el Ejercicio 1
N_CLASES      = 10      # [5] los digitos 0..9
LEARNING_RATE = 1e-3    # [5] paso de Adam
EPOCHS        = 10      # [5] con GPU son ~8 s por epoca

# Fija las semillas para que la corrida sea reproducible.
torch.manual_seed(SEED)
np.random.seed(SEED)

## 2. Los datos `[6]`

`transforms.ToTensor()` hace dos cosas de una sola vez:

1. pasa la imagen PIL a tensor `float32` **dividiendo por 255** — es el
   *"reescalear los datos entre 0 y 1"* del Ejercicio 1;
2. deja la forma en `(C, H, W) = (1, 28, 28)`, que es el orden de ejes de torch
   (**channels first**), al revés que Keras.

El `DataLoader` arma los minibatches y los baraja: es el equivalente de lo que
`model.fit` hace por dentro en Keras cuando le pasás `batch_size`.

In [ ]:
transform = transforms.ToTensor()   # uint8 [0,255] -> float32 [0,1], forma (1,28,28)

# [6] la primera vez descarga ~10 MB en /content/data
train_ds = datasets.MNIST("./data", train=True,  download=True, transform=transform)
test_ds  = datasets.MNIST("./data", train=False, download=True, transform=transform)

# [6] shuffle=True en train para que el orden de los datos no meta correlaciones
# entre pasos de SGD; shuffle=False en test para que las predicciones queden
# alineadas con las etiquetas (lo necesitamos para la matriz de confusion).
# pin_memory acelera la copia CPU -> GPU.
usar_gpu = DEVICE.type == "cuda"
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=usar_gpu)
test_loader  = DataLoader(test_ds,  batch_size=512, shuffle=False,
                          num_workers=2, pin_memory=usar_gpu)

x0, y0 = train_ds[0]
print(f"train: {len(train_ds)} imagenes   test: {len(test_ds)} imagenes")
print(f"forma de una imagen: {tuple(x0.shape)} (C,H,W)")
print(f"rango de x: [{x0.min():.1f}, {x0.max():.1f}]   etiqueta del ejemplo: {int(y0)}")
print(f"batches por epoca: {len(train_loader)} (batch_size={BATCH_SIZE})")

### Inspección visual

Chequeamos a ojo que las imágenes `x` y las etiquetas `y` estén bien apareadas.

In [ ]:
def plot_ejemplos(dataset, n_filas=3, n_cols=6,
                  titulo="MNIST - imagenes y sus etiquetas"):
    fig, axes = plt.subplots(n_filas, n_cols, figsize=(1.5 * n_cols, 1.7 * n_filas))
    for i, ax in enumerate(axes.flat):
        x, y = dataset[i]                       # x: (1,28,28) float en [0,1]
        ax.imshow(x.squeeze().numpy(), cmap="gray")   # squeeze() saca el eje de canal
        ax.set_title(f"y = {int(y)}", fontsize=10)
        ax.axis("off")
    fig.suptitle(titulo)
    fig.tight_layout()
    plt.show()


plot_ejemplos(train_ds)

## 3. El modelo `[1][2][3]`

### `[4]` Keras vs PyTorch

| | **Keras** | **PyTorch** |
|---|---|---|
| Forma del tensor | *channels last*: `(N, 28, 28, 1)` | *channels first*: `(N, 1, 28, 28)` |
| Declarar capas | solo canales de **salida**; los de entrada los infiere sola la primera vez que ve datos (*build* diferido) | entrada **y** salida explícitas: `Conv2d(in_ch, out_ch, k)`, `Linear(in_features, out_features)` |
| Padding | `padding="same"` por nombre | se pone a mano: `padding=1` con `kernel=3` |
| Activaciones | argumento de la capa: `activation="relu"` | funciones aplicadas dentro de `forward` |
| Salida + pérdida | `softmax` + `CategoricalCrossentropy` (one-hot) | `log_softmax` + `NLLLoss` (etiquetas enteras) |
| Entrenamiento | `model.fit(...)` | loop escrito a mano |
| Resumen | `model.summary()` | `print(model)` o `torchinfo.summary` |

### El cálculo de dimensiones

Es la diferencia más incómoda en la práctica: como `Linear` pide
`in_features` de forma explícita, **el número 3136 hay que calcularlo a mano**.

Para una `Conv2d`:

$$H_{out} = \left\lfloor \frac{H_{in} + 2p - k}{s} \right\rfloor + 1$$

Con `kernel=3`, `padding=1`, `stride=1` queda $H_{out} = H_{in}$ — o sea, lo
mismo que el `padding="same"` de Keras. Y para `max_pool2d(2)`:
$H_{out} = H_{in} // 2$.

Recorrido completo:

```
entrada                     (N,  1, 28, 28)
conv1 + relu                (N, 32, 28, 28)     28 -> 28   (padding=1)
max_pool2d(2)               (N, 32, 14, 14)     28 -> 14
conv2 + relu                (N, 64, 14, 14)     14 -> 14
max_pool2d(2)               (N, 64,  7,  7)     14 ->  7
x.view(-1, 64*7*7)          (N, 3136)           aplanado
fc1 + relu                  (N, 128)
fc2                         (N, 10)             logits
log_softmax                 (N, 10)             log-probabilidades
```

### Sobre la cabeza de la red

El Ejercicio 1 cerraba con `GlobalAveragePooling2D` (promediar cada mapa de 7×7
a un solo número). Acá el enunciado pide explícitamente `x.view` + `Linear`, que
es el `Flatten` + `Dense` clásico: **muchísimos más parámetros**
(3136×128 ≈ 400 k contra ≈ 1.3 k de la versión con GAP) y por eso más propenso a
sobreajustar, aunque en MNIST alcanza de sobra.

Otra ventaja de torch que se ve acá: como las activaciones se aplican dentro de
`forward`, el modelo es **código Python común** — se puede meter un `if`, un loop
o un `print` en el medio de la propagación.

In [ ]:
class NN(nn.Module):
    # [1] hereda de nn.Module

    def __init__(self, n_clases=N_CLASES):
        super().__init__()          # [1] indispensable: inicializa nn.Module

        # [2] convolucionales: (canales_entrada, canales_salida, kernel)
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)

        # [2] densas. 64*7*7 = 3136 sale del recorrido de dimensiones de arriba:
        # 64 canales de mapas 7x7 despues de los dos poolings.
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, n_clases)

    def forward(self, x):
        # [3] x llega con forma (N, 1, 28, 28)
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)   # (N,32,28,28) -> (N,32,14,14)
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)   # (N,64,14,14) -> (N,64, 7, 7)

        # [3] x.view aplana los mapas en un vector por muestra. El -1 le dice a
        # torch "deduci vos el tamano del batch"; el otro eje tiene que coincidir
        # con in_features de fc1.
        x = x.view(-1, 64 * 7 * 7)                   # (N, 3136)

        x = F.relu(self.fc1(x))                      # (N, 128)
        x = self.fc2(x)                              # (N, 10)  logits

        # [3] log_softmax sobre el eje de clases -> log-probabilidades,
        # que es lo que espera NLLLoss.
        return F.log_softmax(x, dim=1)


model = NN(N_CLASES).to(DEVICE)     # [7] instanciar y mover al device
print(model)

n_par = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nparametros entrenables: {n_par:,}")

### Verificación de las dimensiones

En vez de creerle a la cuenta de arriba, la comprobamos paso a paso pasando un
batch de prueba por cada capa. Esto es exactamente lo que reemplaza al
`model.summary()` de Keras.

In [ ]:
@torch.no_grad()
def recorrer_dimensiones(model, batch=1):
    x = torch.zeros(batch, 1, 28, 28, device=DEVICE)
    print(f"{'entrada':<22} {tuple(x.shape)}")

    x = F.relu(model.conv1(x));   print(f"{'conv1 + relu':<22} {tuple(x.shape)}")
    x = F.max_pool2d(x, 2);       print(f"{'max_pool2d(2)':<22} {tuple(x.shape)}")
    x = F.relu(model.conv2(x));   print(f"{'conv2 + relu':<22} {tuple(x.shape)}")
    x = F.max_pool2d(x, 2);       print(f"{'max_pool2d(2)':<22} {tuple(x.shape)}")
    x = x.view(-1, 64 * 7 * 7);   print(f"{'x.view(-1, 64*7*7)':<22} {tuple(x.shape)}")
    x = F.relu(model.fc1(x));     print(f"{'fc1 + relu':<22} {tuple(x.shape)}")
    x = model.fc2(x);             print(f"{'fc2 (logits)':<22} {tuple(x.shape)}")
    x = F.log_softmax(x, dim=1);  print(f"{'log_softmax':<22} {tuple(x.shape)}")


recorrer_dimensiones(model, batch=BATCH_SIZE)

# Chequeo rapido: exp(log_softmax) tiene que sumar 1 en cada fila.
with torch.no_grad():
    out = model(torch.zeros(4, 1, 28, 28, device=DEVICE))
print(f"\nsuma de probabilidades por muestra: {out.exp().sum(dim=1).cpu().numpy()}")

El análogo más cercano a `model.summary()` de Keras es `torchinfo`, que no viene
preinstalado pero se instala en un segundo. Esta celda es opcional.

In [ ]:
try:
    from torchinfo import summary
except ImportError:
    %pip install -q torchinfo
    from torchinfo import summary

summary(model, input_size=(BATCH_SIZE, 1, 28, 28))

## 4. Pérdida y optimizador `[7]`

`NLLLoss` (*negative log likelihood*) porque el `forward` ya devuelve
`log_softmax`. Es **equivalente** a `nn.CrossEntropyLoss` aplicada sobre los
logits — esa hace el `log_softmax` por dentro — y equivale a la
`CategoricalCrossentropy` del Ejercicio 1.

La diferencia con Keras es que acá las etiquetas van como **enteros**
(`y = 3`), no como one-hot: es lo mismo que usar
`SparseCategoricalCrossentropy`.

In [ ]:
criterion = nn.NLLLoss()                                             # [7]
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)   # [7]

print(criterion)
print(optimizer)

## 5. El loop de entrenamiento `[8][9]`

Acá está la diferencia más visible con Keras: donde allá alcanzaba con
`model.fit(...)`, en torch el loop se escribe a mano. Los seis pasos que pide el
enunciado, en orden:

1. **propagación hacia adelante** — `out = model(x)`
2. **cálculo de la pérdida** — `loss = criterion(out, y)`
3. **gradientes a 0** — `optimizer.zero_grad()`
4. **cálculo de los gradientes** — `loss.backward()`
5. **llamar al optimizador** — `optimizer.step()`
6. **acumular la pérdida** — `perdida_acum += loss.item() * x.size(0)`

Dos detalles que suelen morder:

- **`zero_grad()` no es opcional.** torch *acumula* los gradientes en
  `.grad`; si no se limpian, los del batch actual se suman a los del anterior y
  el entrenamiento se rompe en silencio.
- **`.item()` saca el número del grafo de autograd.** Si acumulás el tensor
  `loss` directamente, torch mantiene viva toda la historia de operaciones de
  cada batch de la época y se te va la memoria.

In [ ]:
def train_una_epoca(model, loader, criterion, optimizer, epoca=None):
    # [8] un pase completo sobre el train: loop sobre los batches
    model.train()                            # modo entrenamiento
    perdida_acum, correctos, total = 0.0, 0, 0

    barra = tqdm(loader, desc=f"epoca {epoca}", leave=False)
    for x, y in barra:                       # [8] loop sobre batches
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)

        out  = model(x)                      # [9] propagacion hacia adelante
        loss = criterion(out, y)             # [9] calculo de la perdida

        optimizer.zero_grad()                # [9] gradientes a 0
        loss.backward()                      # [9] calculo de los gradientes (backprop)
        optimizer.step()                     # [9] el optimizador actualiza los pesos

        # [9] acumular la perdida. loss es el promedio del batch: lo multiplicamos
        # por el tamano del batch para promediar bien al final (el ultimo batch
        # puede ser mas chico).
        perdida_acum += loss.item() * x.size(0)
        correctos    += (out.argmax(dim=1) == y).sum().item()
        total        += x.size(0)

        barra.set_postfix(loss=f"{perdida_acum/total:.4f}",
                          acc=f"{100*correctos/total:.2f}%")

    return perdida_acum / total, correctos / total


@torch.no_grad()                             # sin autograd: mas rapido y sin memoria extra
def evaluar(model, loader, criterion):
    # [10] perdida, accuracy y predicciones sobre un conjunto
    model.eval()                             # modo evaluacion
    perdida_acum, correctos, total = 0.0, 0, 0
    y_true, y_pred = [], []

    for x, y in loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)

        out  = model(x)
        loss = criterion(out, y)

        pred = out.argmax(dim=1)             # [10] la clase mas probable
        perdida_acum += loss.item() * x.size(0)
        correctos    += (pred == y).sum().item()
        total        += x.size(0)

        y_true.append(y.cpu().numpy())
        y_pred.append(pred.cpu().numpy())

    return (perdida_acum / total, correctos / total,
            np.concatenate(y_true), np.concatenate(y_pred))

In [ ]:
print(f"Entrenando: epocas={EPOCHS}, batch_size={BATCH_SIZE}, "
      f"lr={LEARNING_RATE}, device={DEVICE}\n")

hist = {"loss": [], "acc": [], "test_loss": [], "test_acc": []}

for epoca in range(1, EPOCHS + 1):           # [8] loop sobre epocas
    loss_tr, acc_tr = train_una_epoca(model, train_loader, criterion, optimizer, epoca)
    loss_te, acc_te, _, _ = evaluar(model, test_loader, criterion)

    hist["loss"].append(loss_tr)
    hist["acc"].append(acc_tr)
    hist["test_loss"].append(loss_te)
    hist["test_acc"].append(acc_te)

    print(f"  epoca {epoca:2d}/{EPOCHS}  "
          f"loss={loss_tr:.4f}  acc={100*acc_tr:.2f}%   |   "
          f"test_loss={loss_te:.4f}  test_acc={100*acc_te:.2f}%")

## 6. Evaluación sobre el conjunto de test `[10]`

In [ ]:
loss_test, acc_test, y_true, y_pred = evaluar(model, test_loader, criterion)

print(f"perdida test  = {loss_test:.4f}")
print(f"accuracy test = {acc_test:.4f}  ({100 * acc_test:.2f} %)")
print(f"errores: {int((y_true != y_pred).sum())} sobre {y_true.size}")

In [ ]:
def plot_curvas(hist, titulo="Ejercicio 2 - CNN sobre MNIST (PyTorch)"):
    epocas = np.arange(1, len(hist["loss"]) + 1)
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))

    ax[0].plot(epocas, hist["loss"], "o-", label="train")
    ax[0].plot(epocas, hist["test_loss"], "o-", label="test")
    ax[0].set_xlabel("epoca"); ax[0].set_ylabel("perdida (NLL)")
    ax[0].set_title("Perdida"); ax[0].legend(); ax[0].grid(alpha=0.3)

    ax[1].plot(epocas, 100 * np.array(hist["acc"]), "o-", label="train")
    ax[1].plot(epocas, 100 * np.array(hist["test_acc"]), "o-", label="test")
    ax[1].set_xlabel("epoca"); ax[1].set_ylabel("accuracy [%]")
    ax[1].set_title("Precision"); ax[1].legend(); ax[1].grid(alpha=0.3)

    fig.suptitle(titulo)
    fig.tight_layout()
    plt.show()


plot_curvas(hist)

## 7. Matriz de confusión `[11]`

Fila = clase verdadera, columna = clase predicha. La diagonal son los aciertos;
lo de afuera muestra **qué** confunde la red (típicamente 4↔9, 3↔5, 7↔1), que es
bastante más informativo que el accuracy solo.

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=np.arange(N_CLASES))   # [11]

fig, ax = plt.subplots(figsize=(6.5, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=np.arange(N_CLASES))
disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")   # [11]
ax.set_xlabel("prediccion"); ax.set_ylabel("verdadero")
ax.set_title("Ejercicio 2 - matriz de confusion (test)")
fig.tight_layout()
plt.show()

print("Accuracy por digito:")
for d, a in enumerate(cm.diagonal() / cm.sum(axis=1)):
    print(f"  {d}: {100 * a:.2f} %")

# Los pares que mas se confunden (fuera de la diagonal).
cm_sin_diag = cm.copy()
np.fill_diagonal(cm_sin_diag, 0)
print("\nConfusiones mas frecuentes:")
for k in range(5):
    i, j = np.unravel_index(cm_sin_diag.argmax(), cm_sin_diag.shape)
    print(f"  {cm_sin_diag[i, j]:3d} veces:  un {i} clasificado como {j}")
    cm_sin_diag[i, j] = 0

### Los dígitos que la red erró

Vale la pena mirarlos: buena parte son ambiguos incluso para una persona.

In [ ]:
def plot_errores(dataset, y_true, y_pred, n_max=18,
                 titulo="Ejercicio 2 - ejemplos mal clasificados"):
    idx_mal = np.flatnonzero(y_true != y_pred)[:n_max]
    if idx_mal.size == 0:
        print("no hubo errores en test (!)")
        return

    n_cols  = 6
    n_filas = int(np.ceil(idx_mal.size / n_cols))
    fig, axes = plt.subplots(n_filas, n_cols, figsize=(1.6 * n_cols, 1.9 * n_filas))
    ejes = list(np.atleast_1d(axes).flat)

    for ax in ejes:                          # apaga todos, incluidos los que sobren
        ax.axis("off")
    for ax, i in zip(ejes, idx_mal):
        x, _ = dataset[int(i)]
        ax.imshow(x.squeeze().numpy(), cmap="gray")
        ax.set_title(f"y={y_true[i]}  pred={y_pred[i]}", fontsize=9)

    fig.suptitle(titulo)
    fig.tight_layout()
    plt.show()


plot_errores(test_ds, y_true, y_pred)

## 8. Conclusiones

**Resultado.** La red llega a ~99 % de accuracy en test en 10 épocas, muy
parecido a la versión de Keras del Ejercicio 1. Lo que cambia no es la
performance sino la sintaxis.

**Qué se gana y qué se pierde frente a Keras.**

- Keras es más corto: `model.fit` reemplaza todo el loop de entrenamiento, y no
  hay que declarar dimensiones de entrada ni mover nada a la GPU.
- torch es más explícito: cada paso del entrenamiento está a la vista, y el
  `forward` es código Python común. Eso paga cuando el modelo no es una simple
  pila secuencial de capas — arquitecturas con ramas, pérdidas hechas a medida,
  o cosas que dependen de los datos en tiempo de ejecución.
- El precio de esa explicitud son las trampas: olvidarse de `zero_grad()`, de
  `model.eval()`, del `.to(DEVICE)` o de `.item()` son los cuatro errores
  clásicos, y ninguno da un mensaje de error claro.

**Para seguir probando.** Cosas que se pueden cambiar y ver qué pasa:

- reemplazar la cabeza `x.view` + `Linear` por
  `nn.AdaptiveAvgPool2d(1)` (el `GlobalAveragePooling2D` del Ejercicio 1) y
  comparar cantidad de parámetros contra accuracy;
- agregar `nn.Dropout` o `nn.BatchNorm2d` entre las convoluciones;
- usar `nn.CrossEntropyLoss` sobre los logits en vez de `log_softmax` +
  `NLLLoss`, y verificar que da lo mismo;
- data augmentation con `transforms.RandomAffine(degrees=10, translate=(0.1, 0.1))`.